In [ ]:
import osmnx as ox
import os
import matplotlib.pyplot as plt   # 请确保已安装 matplotlib：pip install matplotlib

# ==================== 【关键】使用 ox.settings 配置（v2+ 必须这样写） ====================
print(f"当前 OSMnx 版本: {ox.__version__}")

ox.settings.use_cache = True
ox.settings.log_console = True
ox.settings.useful_tags_way = []       # 保留路段的【所有】属性
ox.settings.useful_tags_node = []      # 保留节点的【所有】属性

# ==================== 下载上海市杨浦区路网 ====================
place_name = "杨浦区, 上海市, 中国"

G = ox.graph_from_place(
    place_name,
    network_type="all",
    simplify=True,
    retain_all=False
)

# ==================== 强制确保坐标系为 WGS84 ====================
print(f"原始下载 CRS: {G.graph.get('crs')}")
if G.graph.get('crs') != 'EPSG:4326':
    G = ox.project_graph(G, to_crs='EPSG:4326')
    print("✅ 已投影到 WGS84 (EPSG:4326)")
else:
    print("✅ 已是 WGS84 坐标系，无需投影")

# ==================== 转为 GeoDataFrame 并打印属性 ====================
nodes, edges = ox.graph_to_gdfs(G, nodes=True, edges=True)

print(f"\n节点数量: {len(nodes):,} 个")
print(f"路段数量: {len(edges):,} 条")

print("\n=== Nodes 属性列 ===")
print(sorted(nodes.columns.tolist()))
print("\n=== Edges 属性列 ===")
print(sorted(edges.columns.tolist()))

# ==================== 通用修复：将所有对象列中的列表转为字符串（逗号分隔） ====================
def convert_list_columns_to_str(gdf):
    """
    将 GeoDataFrame 中所有 object 类型的列，如果值包含列表，统一转为逗号分隔的字符串。
    否则 Parquet 无法处理混合标量/列表的列。
    """
    for col in gdf.columns:
        # 仅处理 object 类型列（排除 geometry 列）
        if gdf[col].dtype == 'object' and col != gdf.geometry.name:
            # 检查是否有任何值包含列表
            has_list = any(isinstance(x, list) for x in gdf[col] if x is not None)
            if has_list:
                print(f"   → 转换列 '{col}' 中的列表值为字符串")
                gdf[col] = gdf[col].apply(
                    lambda x: ','.join(map(str, x)) if isinstance(x, list) else str(x) if x is not None else None
                )
    return gdf

edges = convert_list_columns_to_str(edges)
# 如果 nodes 也有类似问题（通常不会），可同样处理：nodes = convert_list_columns_to_str(nodes)

# ==================== 保存六份数据 ====================
save_folder = r"D:\AI\Dataset-urban plan\street road"
os.makedirs(save_folder, exist_ok=True)
base_name = "yangpu"

ox.save_graphml(G, filepath=os.path.join(save_folder, f"{base_name}_full_network_wgs84.graphml"))

nodes.to_file(os.path.join(save_folder, f"{base_name}_nodes_wgs84.geojson"), driver="GeoJSON")
nodes.to_parquet(os.path.join(save_folder, f"{base_name}_nodes_wgs84.parquet"))

edges.to_file(os.path.join(save_folder, f"{base_name}_edges_wgs84.geojson"), driver="GeoJSON")
edges.to_parquet(os.path.join(save_folder, f"{base_name}_edges_wgs84.parquet"))

print("\n✅ 所有文件保存完成！路径如下：")
print(f"   • {base_name}_full_network_wgs84.graphml")
print(f"   • {base_name}_nodes_wgs84.geojson / .parquet")
print(f"   • {base_name}_edges_wgs84.geojson / .parquet")

# ==================== 可视化展示 ====================
print("正在生成路网可视化...")
fig, ax = ox.plot_graph(
    G,
    node_size=0,
    edge_color="#1f78b4",
    edge_linewidth=1.0,
    bgcolor="white",
    figsize=(15, 12)
)
plt.title("上海市杨浦区街道路网 (WGS84 坐标系)", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd
import os

# 路径换成你实际保存的位置
folder = r"D:\AI\Dataset-urban plan\street road"

# 读取节点和路段
nodes = gpd.read_file(os.path.join(folder, "yangpu_nodes_wgs84.geojson"))
edges = gpd.read_file(os.path.join(folder, "yangpu_edges_wgs84.geojson"))

# 打印属性列（geometry 列也会显示，但你可以排除）
print("=== 节点属性字段 ===")
print(nodes.columns.tolist())

print("\n=== 路段属性字段 ===")
print(edges.columns.tolist())

# 如果想查看每个字段的数据类型
print("\n节点字段类型：")
print(nodes.dtypes)
print("\n路段字段类型：")
print(edges.dtypes)

In [ ]:
# 对路网数据进行标准化处理

from pathlib import Path
import json

import geopandas as gpd
import pandas as pd
import numpy as np
import networkx as nx


INPUT_DIR = Path(r"D:\AI\Dataset-urban plan\street road")
OUTPUT_DIR = Path(r"D:\AI\urban_renewal_data\processed\road")

NODES_PATH = INPUT_DIR / "yangpu_nodes_wgs84.parquet"
EDGES_PATH = INPUT_DIR / "yangpu_edges_wgs84.parquet"

OUT_NODES_PARQUET = OUTPUT_DIR / "road_nodes_clean.parquet"
OUT_NODES_GEOJSON = OUTPUT_DIR / "road_nodes_clean.geojson"
OUT_EDGES_PARQUET = OUTPUT_DIR / "road_edges_clean.parquet"
OUT_EDGES_GEOJSON = OUTPUT_DIR / "road_edges_clean.geojson"
OUT_GRAPHML = OUTPUT_DIR / "walk_bike_network.graphml"
OUT_REPORT = OUTPUT_DIR / "road_quality_report.md"

WALK_SPEED_KMH = 4.8
BIKE_SPEED_KMH = 12.0
PROJECT_CRS = "EPSG:32651"


def _safe_reset_index(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """安全重置命名索引(支持单层 Index 和 MultiIndex),
    并自动 drop 与索引层同名的列,避免 reset_index 时报
    'cannot insert <name>, already exists'。"""
    if isinstance(gdf.index, pd.MultiIndex):
        named = [n for n in gdf.index.names if n is not None]
    else:
        named = [gdf.index.name] if gdf.index.name is not None else []
    if not named:
        return gdf
    cols_to_drop = [n for n in named if n in gdf.columns]
    if cols_to_drop:
        gdf = gdf.drop(columns=cols_to_drop)
    return gdf.reset_index()


def ensure_columns_from_index(gdf: gpd.GeoDataFrame, cols: list[str]) -> gpd.GeoDataFrame:
    gdf = gdf.copy()
    for col in cols:
        if col not in gdf.columns and col in gdf.index.names:
            gdf[col] = gdf.index.get_level_values(col)
    return gdf


def to_bool_str(value) -> str:
    if pd.isna(value):
        return "unknown"
    return str(value)


def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    nodes = gpd.read_parquet(NODES_PATH)
    edges = gpd.read_parquet(EDGES_PATH)

    # 关键修正:同时处理单层 Index(nodes 的 osmid)和 MultiIndex(edges 的 u,v,key)
    nodes = _safe_reset_index(nodes)
    edges = _safe_reset_index(edges)

    if nodes.crs is None:
        nodes = nodes.set_crs("EPSG:4326")
    else:
        nodes = nodes.to_crs("EPSG:4326")

    if edges.crs is None:
        edges = edges.set_crs("EPSG:4326")
    else:
        edges = edges.to_crs("EPSG:4326")

    # 在 _safe_reset_index 之后,u/v/key/osmid 已经是列,这一步是兜底
    nodes = ensure_columns_from_index(nodes, ["osmid"])
    edges = ensure_columns_from_index(edges, ["u", "v", "key"])

    if "osmid" not in nodes.columns:
        nodes["osmid"] = nodes.index.astype(str)

    if "x" not in nodes.columns:
        nodes["x"] = nodes.geometry.x

    if "y" not in nodes.columns:
        nodes["y"] = nodes.geometry.y

    if "length" in edges.columns:
        edges["length_m"] = pd.to_numeric(edges["length"], errors="coerce")
    else:
        edges["length_m"] = np.nan

    missing_length = edges["length_m"].isna() | (edges["length_m"] <= 0)
    if missing_length.any():
        edges_proj = edges.to_crs(PROJECT_CRS)
        edges.loc[missing_length, "length_m"] = edges_proj.loc[missing_length].geometry.length

    edges = edges[edges["length_m"].notna() & (edges["length_m"] > 0)].copy()

    nodes["node_id"] = nodes["osmid"].astype(str)
    nodes["lon_wgs84"] = pd.to_numeric(nodes["x"], errors="coerce")
    nodes["lat_wgs84"] = pd.to_numeric(nodes["y"], errors="coerce")

    edge_ends = pd.concat(
        [edges["u"].astype(str), edges["v"].astype(str)],
        ignore_index=True,
    )
    degree_counts = edge_ends.value_counts().to_dict()

    nodes["degree_calc"] = nodes["node_id"].map(degree_counts).fillna(0).astype(int)

    if "street_count" in nodes.columns:
        nodes["street_count"] = pd.to_numeric(nodes["street_count"], errors="coerce").fillna(0).astype(int)
    else:
        nodes["street_count"] = nodes["degree_calc"]

    nodes["is_intersection"] = (nodes["street_count"] >= 3) | (nodes["degree_calc"] >= 3)

    nodes_clean = nodes[
        [
            "node_id", "osmid", "lon_wgs84", "lat_wgs84",
            "street_count", "degree_calc", "is_intersection", "geometry",
        ]
    ].copy()
    # 再保险一次:确保 nodes_clean 是干净的 RangeIndex
    nodes_clean = nodes_clean.reset_index(drop=True)

    edges["u"] = edges["u"].astype(str)
    edges["v"] = edges["v"].astype(str)
    edges["key"] = edges["key"].astype(str)

    edges["edge_id"] = edges.apply(
        lambda r: f"road_edge_{r['u']}_{r['v']}_{r['key']}",
        axis=1,
    )

    edges["from_node"] = edges["u"]
    edges["to_node"] = edges["v"]
    edges["walkable"] = True
    edges["bikeable"] = True
    edges["walk_speed_kmh"] = WALK_SPEED_KMH
    edges["bike_speed_kmh"] = BIKE_SPEED_KMH
    edges["walk_time_min"] = edges["length_m"] / (WALK_SPEED_KMH * 1000 / 60)
    edges["bike_time_min"] = edges["length_m"] / (BIKE_SPEED_KMH * 1000 / 60)
    edges["oneway_original"] = edges["oneway"].apply(to_bool_str) if "oneway" in edges.columns else "unknown"
    edges["reversed_original"] = edges["reversed"].apply(to_bool_str) if "reversed" in edges.columns else "unknown"
    edges["road_name"] = "unknown"
    edges["road_type"] = "unknown"
    edges["source"] = "osmnx_openstreetmap"
    edges["source_crs"] = "WGS84"
    edges["analysis_crs"] = "WGS84"

    if "osmid" not in edges.columns:
        edges["osmid"] = ""

    edges["osmid"] = edges["osmid"].astype(str)

    edges_clean = edges[
        [
            "edge_id", "from_node", "to_node", "u", "v", "key", "osmid",
            "road_name", "road_type", "length_m",
            "walkable", "bikeable",
            "walk_speed_kmh", "bike_speed_kmh",
            "walk_time_min", "bike_time_min",
            "oneway_original", "reversed_original",
            "source", "source_crs", "analysis_crs",
            "geometry",
        ]
    ].copy()
    # 再保险一次:确保 edges_clean 是干净的 RangeIndex
    edges_clean = edges_clean.reset_index(drop=True)

    nodes_clean.to_parquet(OUT_NODES_PARQUET, index=False)
    nodes_clean.to_file(OUT_NODES_GEOJSON, driver="GeoJSON", encoding="utf-8")

    edges_clean.to_parquet(OUT_EDGES_PARQUET, index=False)
    edges_clean.to_file(OUT_EDGES_GEOJSON, driver="GeoJSON", encoding="utf-8")

    G = nx.MultiDiGraph()
    G.graph["crs"] = "EPSG:4326"

    for _, row in nodes_clean.iterrows():
        G.add_node(
            row["node_id"],
            x=float(row["lon_wgs84"]),
            y=float(row["lat_wgs84"]),
            osmid=str(row["osmid"]),
            street_count=int(row["street_count"]),
            degree_calc=int(row["degree_calc"]),
            is_intersection=int(bool(row["is_intersection"])),
        )

    for _, row in edges_clean.iterrows():
        attrs = {
            "edge_id": row["edge_id"],
            "osmid": str(row["osmid"]),
            "length_m": float(row["length_m"]),
            "walkable": 1,
            "bikeable": 1,
            "walk_time_min": float(row["walk_time_min"]),
            "bike_time_min": float(row["bike_time_min"]),
            "road_name": str(row["road_name"]),
            "road_type": str(row["road_type"]),
            "oneway_original": str(row["oneway_original"]),
            "geometry_wkt": row["geometry"].wkt if row["geometry"] is not None else "",
        }

        u = str(row["from_node"])
        v = str(row["to_node"])

        if u in G and v in G:
            G.add_edge(u, v, key=f"{row['edge_id']}_fwd", **attrs)
            G.add_edge(v, u, key=f"{row['edge_id']}_rev", **attrs)

    nx.write_graphml(G, OUT_GRAPHML)

    total_length_km = edges_clean["length_m"].sum() / 1000
    avg_length_m = edges_clean["length_m"].mean()
    intersection_count = int(nodes_clean["is_intersection"].sum())

    report = f"""# 杨浦区路网数据处理报告

## 输入

- 节点文件:`{NODES_PATH}`
- 路段文件:`{EDGES_PATH}`

## 输出

- `road_nodes_clean.parquet`
- `road_nodes_clean.geojson`
- `road_edges_clean.parquet`
- `road_edges_clean.geojson`
- `walk_bike_network.graphml`

## 统计

| 指标 | 数值 |
|---|---:|
| 节点数量 | {len(nodes_clean)} |
| 路段数量 | {len(edges_clean)} |
| 路口节点数量 | {intersection_count} |
| 总路段长度 km | {total_length_km:.2f} |
| 平均路段长度 m | {avg_length_m:.2f} |
| 默认步行速度 km/h | {WALK_SPEED_KMH} |
| 默认骑行速度 km/h | {BIKE_SPEED_KMH} |

## 说明

当前 MVP 阶段默认所有路段均可步行、可骑行,并将路网构建为双向网络。
由于原始 OSMnx 数据未保留道路名称和道路等级,本阶段将 `road_name` 和 `road_type` 暂记为 `unknown`。
"""

    OUT_REPORT.write_text(report, encoding="utf-8")

    print("完成")
    print(OUT_NODES_PARQUET)
    print(OUT_EDGES_PARQUET)
    print(OUT_GRAPHML)
    print(OUT_REPORT)


if __name__ == "__main__":
    main()

In [ ]:
# 路网查询测试

from pathlib import Path
import math

import geopandas as gpd
import networkx as nx
import pandas as pd


ROAD_DIR = Path(r"D:\AI\urban_renewal_data\processed\road")
NODES_PATH = ROAD_DIR / "road_nodes_clean.parquet"
GRAPH_PATH = ROAD_DIR / "walk_bike_network.graphml"

# 示例坐标，可改成任意杨浦区内两个点
ORIGIN_LON = 121.509
ORIGIN_LAT = 31.286
DEST_LON = 121.523
DEST_LAT = 31.280


def haversine_m(lon1, lat1, lon2, lat2):
    r = 6371000
    p1 = math.radians(lat1)
    p2 = math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)

    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))


def nearest_node(nodes: gpd.GeoDataFrame, lon: float, lat: float) -> str:
    temp = nodes.copy()
    temp["dist_m"] = temp.apply(
        lambda r: haversine_m(lon, lat, r["lon_wgs84"], r["lat_wgs84"]),
        axis=1,
    )
    return str(temp.sort_values("dist_m").iloc[0]["node_id"])


def main():
    nodes = gpd.read_parquet(NODES_PATH)
    G = nx.read_graphml(GRAPH_PATH)

    origin_node = nearest_node(nodes, ORIGIN_LON, ORIGIN_LAT)
    dest_node = nearest_node(nodes, DEST_LON, DEST_LAT)

    route = nx.shortest_path(
        G,
        source=origin_node,
        target=dest_node,
        weight="walk_time_min",
    )

    length_m = 0
    walk_time_min = 0

    for u, v in zip(route[:-1], route[1:]):
        edge_data = G.get_edge_data(u, v)
        first_edge = list(edge_data.values())[0]
        length_m += float(first_edge["length_m"])
        walk_time_min += float(first_edge["walk_time_min"])

    print("路径测试完成")
    print(f"起点节点: {origin_node}")
    print(f"终点节点: {dest_node}")
    print(f"经过节点数: {len(route)}")
    print(f"路径长度: {length_m:.1f} m")
    print(f"步行时间: {walk_time_min:.1f} min")


if __name__ == "__main__":
    main()